# 739. Daily Temperatures

**Difficulty**: Medium  
**Topics**: Array, Stack, Monotonic Stack  
**Link**: [LeetCode 739](https://leetcode.com/problems/daily-temperatures/)

---

## Problem Statement

Given an array of integers `temperatures` where `temperatures[i]` represents the daily temperature on day `i`, return an array `answer` such that `answer[i]` is the number of days you have to wait after day `i` to get a warmer temperature. If there is no future day for which this is possible, keep `answer[i] == 0` instead.

### Examples

**Example 1:**
```
Input: temperatures = [73,74,75,71,69,72,76,73]
Output: [1,1,4,2,1,1,0,0]
```

**Example 2:**
```
Input: temperatures = [30,40,50,60]
Output: [1,1,1,0]
```

**Example 3:**
```
Input: temperatures = [30,60,90]
Output: [1,1,0]
```

### Constraints

- `1 <= temperatures.length <= 10^5`
- `30 <= temperatures[i] <= 100`


---

## Approach 1: Monotonic Stack

### Intuition

For each day, we want to know the **next future day with a higher temperature**.

If we try to scan to the right for every day, we repeat a lot of comparisons. The monotonic stack avoids that repeated work by keeping only the days that are still waiting for a warmer answer.

A monotonic stack is a perfect fit because it lets us keep track of unresolved days in an organized way.

We store **indices**, not temperatures, because the output needs the distance between days, not just the warmer value itself.

### Stack invariant

The stack stores indices whose temperatures are in **decreasing order** from bottom to top.

That means:

- the top of the stack is the most recent unresolved day
- every day in the stack is still waiting for a warmer temperature
- if the current temperature is warmer than the top, it will also resolve that top day immediately

### Why it works

When we see a new temperature that is warmer than the temperature at the top of the stack, we have found the answer for that earlier day.

The number of waiting days is simply:

```
current_index - previous_index
```

If the current day is **not** warmer, we push it onto the stack and wait for a future warmer day.

Each index is pushed once and popped once at most, which is why the total work is linear.

### Complexity
- **Time**: O(n)
- **Space**: O(n)


In [ ]:
def daily_temperatures_stack(temperatures: list[int]) -> list[int]:
    n = len(temperatures)
    answer = [0] * n
    stack = []

    for i, temp in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < temp:
            prev_idx = stack.pop()
            answer[prev_idx] = i - prev_idx
        stack.append(i)

    return answer

test_cases = [
    ([73,74,75,71,69,72,76,73], [1,1,4,2,1,1,0,0]),
    ([30,40,50,60], [1,1,1,0]),
    ([30,60,90], [1,1,0]),
    ([90,80,70], [0,0,0]),
    ([70], [0]),
]

for temperatures, expected in test_cases:
    result = daily_temperatures_stack(temperatures)
    print(f"{temperatures} -> {result} (expected {expected})")


---

## Approach 2: Reverse Traversal with Jump Table

### Intuition

Instead of using a stack, we can scan from right to left and use already computed answers to jump forward quickly.

For each day `i`, we check the next day `j = i + 1`:

- if `temperatures[j]` is warmer, we are done
- otherwise, use `answer[j]` to jump ahead to the next candidate

### Why it works

Suppose day `j` is not warm enough for day `i`. If `answer[j] = d`, then day `j + d` is the first warmer day for `j`.

That means every day between `j` and `j + d` is **not warmer than `temperatures[j]`**, so those days also cannot help day `i` when `temperatures[j] <= temperatures[i]`.

So instead of checking every intermediate day one by one, we can jump directly to the next useful candidate.

This makes the method efficient, even though it may look like nested looping at first glance.

### When this approach is useful

This approach is a good alternative when:

- you are comfortable reasoning from right to left
- you want to avoid an explicit stack
- you already have answers that can help skip work

### Complexity
- **Time**: O(n) amortized
- **Space**: O(1) extra space beyond output


In [ ]:
def daily_temperatures_jump(temperatures: list[int]) -> list[int]:
    n = len(temperatures)
    answer = [0] * n

    for i in range(n - 2, -1, -1):
        j = i + 1

        while j < n and temperatures[j] <= temperatures[i] and answer[j] > 0:
            j += answer[j]

        if j < n and temperatures[j] > temperatures[i]:
            answer[i] = j - i

    return answer

for temperatures, expected in test_cases:
    result = daily_temperatures_jump(temperatures)
    print(f"{temperatures} -> {result} (expected {expected})")


---

## Approach 3: Brute Force

### Intuition

For each day, scan forward until a warmer temperature is found.

### Why include it

This approach is simple and useful for understanding the problem, but it is too slow for large inputs.

### Complexity
- **Time**: O(n^2)
- **Space**: O(1) extra space


In [ ]:
def daily_temperatures_bruteforce(temperatures: list[int]) -> list[int]:
    n = len(temperatures)
    answer = [0] * n

    for i in range(n):
        for j in range(i + 1, n):
            if temperatures[j] > temperatures[i]:
                answer[i] = j - i
                break

    return answer

print(daily_temperatures_bruteforce([73,74,75,71,69,72,76,73]))


---

## Walkthrough: Monotonic Stack

Input: `temperatures = [73,74,75,71,69,72,76,73]`

Think of the stack as a list of days that are still waiting for a warmer future day.

When a new temperature arrives:

- pop all colder days from the top
- fill in their waiting times
- push the current day because it may help resolve future days

| Day | Temp | Stack Before | Action | Stack After | Answer Update |
|-----|------|--------------|--------|-------------|---------------|
| 0 | 73 | [] | push 0 | [0] | - |
| 1 | 74 | [0] | pop 0, answer[0] = 1 | [] | answer[0] = 1 |
| 1 | 74 | [] | push 1 | [1] | - |
| 2 | 75 | [1] | pop 1, answer[1] = 1 | [] | answer[1] = 1 |
| 2 | 75 | [] | push 2 | [2] | - |
| 3 | 71 | [2] | push 3 | [2,3] | - |
| 4 | 69 | [2,3] | push 4 | [2,3,4] | - |
| 5 | 72 | [2,3,4] | pop 4, answer[4] = 1 | [2,3] | answer[4] = 1 |
| 5 | 72 | [2,3] | pop 3, answer[3] = 2 | [2] | answer[3] = 2 |
| 5 | 72 | [2] | push 5 | [2,5] | - |
| 6 | 76 | [2,5] | pop 5, answer[5] = 1 | [2] | answer[5] = 1 |
| 6 | 76 | [2] | pop 2, answer[2] = 4 | [] | answer[2] = 4 |
| 6 | 76 | [] | push 6 | [6] | - |
| 7 | 73 | [6] | push 7 | [6,7] | - |

Notice that days `6` and `7` stay in the stack at the end. That means no warmer day exists after them, so their answers remain `0`.

Final answer: `[1,1,4,2,1,1,0,0]`


In [ ]:
def daily_temperatures_verbose(temperatures: list[int]) -> list[int]:
    answer = [0] * len(temperatures)
    stack = []

    for i, temp in enumerate(temperatures):
        print(f"Day {i}, temp = {temp}")
        print(f"  stack before: {stack}")

        while stack and temperatures[stack[-1]] < temp:
            prev = stack.pop()
            answer[prev] = i - prev
            print(f"  pop day {prev} ({temperatures[prev]}), set answer[{prev}] = {i - prev}")

        stack.append(i)
        print(f"  stack after : {stack}")
        print(f"  answer      : {answer}")
        print()

    return answer

daily_temperatures_verbose([73,74,75,71,69,72,76,73])


---

## Comparison

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Monotonic Stack** | O(n) | O(n) | Most standard, easy to reason about | Uses extra stack |
| **Reverse Traversal + Jump Table** | O(n) amortized | O(1) extra | Clever, no extra stack | Less intuitive |
| **Brute Force** | O(n^2) | O(1) extra | Easiest to understand | Too slow for large input |

**Best choice**: **Monotonic Stack** because it is clear, fast, and matches the pattern directly.


---

## Edge Cases

| Case | Input | Output | Reason |
|------|-------|--------|--------|
| Single day | `[70]` | `[0]` | No future day exists |
| Strictly increasing | `[30,40,50]` | `[1,1,0]` | Each day waits one day except last |
| Strictly decreasing | `[90,80,70]` | `[0,0,0]` | No warmer day exists |
| All equal | `[70,70,70]` | `[0,0,0]` | Need strictly warmer, not equal |
| Warm day far away | `[70,60,50,80]` | `[3,2,1,0]` | Answer may be several days later |


---

## Common Mistakes

1. **Using `<=` instead of `<` in the stack condition**
   ```python
   # WRONG if equal temperatures should not count as warmer
   while stack and temperatures[stack[-1]] <= temp:
   
   # CORRECT
   while stack and temperatures[stack[-1]] < temp:
   ```

2. **Storing temperatures instead of indices**
   ```python
   # WRONG: cannot compute day distance easily
   stack.append(temp)
   
   # CORRECT
   stack.append(i)
   ```

3. **Forgetting unanswered days remain 0**
   
   Any indices left in the stack after the loop have no warmer day in the future.


---

## Related Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [496. Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/) | Easy | Monotonic stack |
| [503. Next Greater Element II](https://leetcode.com/problems/next-greater-element-ii/) | Medium | Circular monotonic stack |
| [84. Largest Rectangle in Histogram](https://leetcode.com/problems/largest-rectangle-in-histogram/) | Hard | Monotonic boundaries |
| [901. Online Stock Span](https://leetcode.com/problems/online-stock-span/) | Medium | Monotonic stack with spans |


---

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Monotonic decreasing stack** | When searching for the next greater element |
| **Store indices** | When answers depend on positions or distances |
| **Pop while current is better** | When the current element resolves pending answers |
| **Leave defaults in place** | When unresolved elements should keep 0 or -1 |

A useful way to recognize this problem is to rephrase it as:

> For each index, find the next index to the right with a greater value.

That wording makes it clear that this is a classic **next greater element to the right** problem.

Once you see that pattern, the monotonic stack solution becomes the natural choice.
